# Survival analysis — Customer Health Intelligence

This notebook walks through **Phase 2** of the ML pipeline:

1. **Feature engineering** — usage trends, payment health, support sentiment
2. **Cox proportional hazards** — predict *when* customers churn (not just if)
3. **Kaplan–Meier** — overall retention curve for the portfolio

Run from project root with `venv` activated. Requires `DATABASE_URL` in `.env.local` or CSVs in `data/synthetic/`.

In [ ]:
import sys
from pathlib import Path

ROOT = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
sys.path.insert(0, str(ROOT))

from ml.data_loader import load_training_data
from ml.feature_engineering import build_customer_features, prepare_cox_matrix
from ml.survival_model import fit_kaplan_meier, train_survival_models
from ml.train_models import run_training

## Cox model — intuition

Cox regression estimates how covariates (MRR, login trend, payment failures) affect the **hazard rate** — the instantaneous risk of churn at time *t*.

- **Duration**: days from signup to churn (or censoring at snapshot date)
- **Event**: 1 if churned, 0 if still active (right-censored)

We predict **forward-looking** risk using `conditional_after=tenure_days`: "Given the customer has survived this long, what's the probability they churn in the next 30/90 days?"

In [ ]:
features, predictions, metrics = run_training(prefer_postgres=True)
print(f"C-index: {metrics['concordance_index']}")
print(f"High 30d risk (>0.6): {(predictions['churn_risk_30d'] > 0.6).sum()}")
features.head()

## Kaplan–Meier curve

Non-parametric estimate of portfolio retention over time (no covariates).

In [ ]:
import matplotlib.pyplot as plt

kmf = fit_kaplan_meier(features)
ax = kmf.plot()
ax.set_title("Portfolio retention (Kaplan–Meier)")
ax.set_xlabel("Days since signup")
ax.set_ylabel("Survival probability")
plt.show()

## Upload predictions

```bash
python scripts/run_ml_pipeline.py --replace
```